# CPU vs GPU consistency checks

This notebook is mainly used to validate the GPU implementations against their CPU reference counterparts (sanity check)

In [ ]:
import sys, os, time
sys.path.append(os.path.abspath('..'))

import time
import numpy as np
import pandas as pd
from numba import cuda

from simulation.simulation import (
    simulate_outer_market_paths,
    simulate_nested_cva as simulate_nested_cva_gpu,
    simulate_nested_cva_swaptions_gpu,
)
from simulation.simulation_cpu import (
    simulate_nested_cva as simulate_nested_cva_cpu,
    simulate_nested_cva_swaptions,
)
from products.irs.gpu import price_irs as price_irs_device
from products.irs.cpu import price_irs as price_irs_cpu
from products.swaption.cpu import price_bermudan_swaption

In [ ]:
# Compare price_irs (GPU kernel function) vs price_irs (cpu primary version) over a (t, short rate r) grid
@cuda.jit
def _price_irs_kernel(t_arr, r_arr, first_reset, reset_freq, num_resets,
                      swap_rate, a, b, sigma, out):
    i = cuda.grid(1)
    if i >= t_arr.size:
        return
    out[i] = price_irs_device(
        t_arr[i], r_arr[i], r_arr[i],
        first_reset, reset_freq, num_resets,
        swap_rate, a, b, sigma, False, 1e-6,
    )


swap = {'first_reset': 0.5, 'reset_freq': 0.5, 'num_resets': 10, 'swap_rate': 0.03}
a, b, sigma = 0.3, 0.03, 0.01

t_grid = [0.0, 0.5, 1.0, 2.0, 3.0]
r_grid = [0.005, 0.01, 0.02, 0.03, 0.04, 0.05]
tt, rr = np.meshgrid(t_grid, r_grid, indexing='ij')
t_flat = tt.ravel().astype(np.float32)
r_flat = rr.ravel().astype(np.float32)
n = t_flat.size

out_gpu = cuda.device_array(n, dtype=np.float32)
threads = 32
blocks = (n + threads - 1) // threads
_price_irs_kernel[blocks, threads](
    cuda.to_device(t_flat), cuda.to_device(r_flat),
    swap['first_reset'], swap['reset_freq'], swap['num_resets'],
    swap['swap_rate'], a, b, sigma,
    out_gpu,
)
cuda.synchronize()
gpu_prices = out_gpu.copy_to_host()

rows = []
for idx in range(n):
    t, r = float(t_flat[idx]), float(r_flat[idx])
    p_cpu = price_irs_cpu(
        t=t, r=r, r_last_fixing=r,
        first_reset=swap['first_reset'],
        reset_freq=swap['reset_freq'],
        num_resets=swap['num_resets'],
        swap_rate=swap['swap_rate'],
        a=a, b=b, sigma=sigma,
    )
    p_gpu = float(gpu_prices[idx])
    rows.append([t, r, f"{p_cpu:.6f}", f"{p_gpu:.6f}", f"{abs(p_cpu - p_gpu):.2e}"])

df = pd.DataFrame(rows, columns=["t", "r", "CPU", "GPU", "|CPU-GPU|"])
print(df.to_string(index=False))

  t     r       CPU       GPU |CPU-GPU|
0.0 0.005 -0.066702 -0.066702  1.62e-09
0.0 0.010 -0.055959 -0.055959  1.29e-09
0.0 0.020 -0.034946 -0.034946  5.71e-10
0.0 0.030 -0.014553 -0.014553  2.00e-10
0.0 0.040  0.005238  0.005238  1.55e-10
0.0 0.050  0.024441  0.024441  7.09e-10
0.5 0.005 -0.060470 -0.060470  9.74e-10
0.5 0.010 -0.048004 -0.048004  8.33e-10
0.5 0.020 -0.023522 -0.023522  7.92e-10
0.5 0.030  0.000376  0.000376  4.89e-12
0.5 0.040  0.023703  0.023703  1.73e-10
0.5 0.050  0.046472  0.046472  7.15e-10
1.0 0.005 -0.068748 -0.068748  1.70e-09
1.0 0.010 -0.054642 -0.054642  1.55e-09
1.0 0.020 -0.026812 -0.026812  2.10e-10
1.0 0.030  0.000516  0.000516  1.42e-11
1.0 0.040  0.027355  0.027355  3.61e-11
1.0 0.050  0.053716  0.053716  1.28e-09
2.0 0.005 -0.060324 -0.060324  8.76e-10
2.0 0.010 -0.047972 -0.047972  5.14e-11
2.0 0.020 -0.023543 -0.023543  8.96e-10
2.0 0.030  0.000527  0.000527  2.64e-11
2.0 0.040  0.024244  0.024244  2.69e-10
2.0 0.050  0.047617  0.047617  1.00e-09


/opt/python/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [ ]:
# Compare full nested CVA on swaps and Bermudan swaptions (GPU vs CPU) on a shared outer sample

dT = 0.1
num_substeps = 5
dt = dT / num_substeps
T_horizon = 10.0
num_steps_total = round(T_horizon / dT)

a, b, sigma = 0.3, 0.03, 0.01
k, theta, xi = 0.5, 0.015, 0.01
rho = 0.0
diff_params = (a, b, sigma, k, theta, xi)
r_0, gamma_0 = 0.01, 0.015

num_outer = 256
N_b = 3
reset_freq = 0.5

np.random.seed(0)
num_irs = 20
notional_irs = 10000. * (np.random.choice((-1, 1), num_irs) * np.random.choice(range(1, 11), num_irs))
irs = [{
    'first_reset': reset_freq, 'reset_freq': reset_freq,
    'num_resets': int(np.random.randint(6, 21)),
    'notional': float(notional_irs[i]),
    'swap_rate': float(np.random.uniform(0.005, 0.05)),
} for i in range(num_irs)]

np.random.seed(1)
num_swaptions = 5
swap_types = np.random.choice((-1, 1), num_swaptions, p=(0.25, 0.75))
notional_swpt = 10000. * np.random.choice(range(1, 11), num_swaptions)
swaptions = [{
    'first_reset': float(np.random.choice([1.0, 1.5, 2.0])),
    'reset_freq': reset_freq,
    'num_resets': int(np.random.randint(10, 17)),
    'notional': float(notional_swpt[i]),
    'swap_rate': float(np.random.uniform(0.005, 0.05)),
    'swap_type': int(swap_types[i]),
} for i in range(num_swaptions)]

fixing_window_size = max(int((reset_freq + dt) / dT), 1)

rng_outer = np.random.default_rng(42)
X, default_step, rate_integral_path, gamma_integral_path = simulate_outer_market_paths(
    num_outer_paths=num_outer, num_steps_total=num_steps_total,
    num_substeps=num_substeps, dt=dt,
    fixing_window_size=fixing_window_size,
    r_0=r_0, gamma_0=gamma_0,
    diff_params=diff_params, rho=rho, rng=rng_outer,
)
print(f"outer paths simulated | defaults: {(default_step != -1).sum()}/{num_outer}")

outer paths simulated | defaults: 30/256


In [ ]:
# swaps
t_i_idx = round(2.5 / dT)
t_i = t_i_idx * dT
num_steps_nested = num_steps_total - t_i_idx
defaulted = (default_step != -1) & (default_step < t_i_idx)

for indicator, label in [(True, "indicator"), (False, "intensity")]:
    rows = []
    for M_inner in [32, 64, 128]:
        t0 = time.time()
        cva_cpu, _ = simulate_nested_cva_cpu(
            t_i_idx, num_steps_nested, t_i, X, defaulted,
            diff_params, rho, irs, np.random.default_rng(42),
            dt, num_substeps, num_outer, M_inner,
            fixing_window_size, indicator_in_cva=indicator,
        )
        t_cpu = time.time() - t0
        m_cpu = cva_cpu.mean()
        ic_cpu = 1.96 * cva_cpu.std(ddof=1) / np.sqrt(num_outer)

        t0 = time.time()
        cva_gpu, _, _ = simulate_nested_cva_gpu(
            t_i_idx, M_inner, indicator, X, default_step, irs,
            diff_params, rho, dt, num_substeps, num_steps_total,
            num_outer, fixing_window_size, dT, seed=42,
        )
        t_gpu = time.time() - t0
        m_gpu = cva_gpu.mean()
        ic_gpu = 1.96 * cva_gpu.std(ddof=1) / np.sqrt(num_outer)

        diff = abs(m_cpu - m_gpu)
        ok = "OK" if diff < 2 * max(ic_cpu, ic_gpu) else "DIFF"
        rows.append([M_inner, f"{m_cpu:.2f} ±{ic_cpu:.2f}", f"{m_gpu:.2f} ±{ic_gpu:.2f}",
                     f"{diff:.2f}", f"{t_cpu:.2f}", f"{t_gpu:.2f}",
                     f"{t_cpu/t_gpu:.1f}x", ok])
    df = pd.DataFrame(rows, columns=["M_inner", "CPU value", "GPU value",
                                     "|CPU-GPU|", "t CPU (s)", "t GPU (s)", "speedup", "status"])
    print(f"\n CVA swaps ({label}) at t_i=2.5, M_cva={num_outer}, n_irs={num_irs}")
    print(df.to_string(index=False))

/opt/python/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))



 CVA swaps (indicator) at t_i=2.5, M_cva=256, n_irs=20
 M_inner   CPU value   GPU value |CPU-GPU| t CPU (s) t GPU (s) speedup status
      32 17.32 ±6.61 24.74 ±9.33      7.42     82.14      1.31   62.7x     OK
      64 17.53 ±5.67 18.85 ±6.48      1.32    163.58      0.22  727.1x     OK
     128 18.16 ±4.66 20.21 ±5.49      2.04    327.65      0.44  736.4x     OK

 CVA swaps (intensity) at t_i=2.5, M_cva=256, n_irs=20
 M_inner   CPU value   GPU value |CPU-GPU| t CPU (s) t GPU (s) speedup status
      32 18.83 ±4.18 18.63 ±4.10      0.20     85.32      0.11  791.6x     OK
      64 18.87 ±3.90 18.68 ±4.05      0.20    169.77      0.21  803.2x     OK
     128 18.53 ±3.89 19.52 ±4.12      0.98    339.17      0.42  809.4x     OK


In [13]:
# swaptions
for indicator, label in [(True, "indicator"), (False, "intensity")]:
    rows = []
    for M_LS in [16, 32, 64]:
        t0 = time.time()
        cva_cpu, _ = simulate_nested_cva_swaptions(
            swaptions, X, default_step, rate_integral_path, gamma_integral_path,
            diff_params, num_outer, num_steps_total, M_LS,
            dt, dT, N_b, rng=np.random.default_rng(42),
            indicator_in_cva=indicator,
        )
        t_cpu = time.time() - t0
        m_cpu = cva_cpu.mean()
        ic_cpu = 1.96 * cva_cpu.std(ddof=1) / np.sqrt(num_outer)

        t0 = time.time()
        cva_gpu, _ = simulate_nested_cva_swaptions_gpu(
            swaptions, X, default_step, rate_integral_path, gamma_integral_path,
            diff_params, rho, dt, dT, num_substeps, num_steps_total,
            num_outer, M_LS, fixing_window_size,
            indicator_in_cva=indicator, seed=42,
        )
        t_gpu = time.time() - t0
        m_gpu = cva_gpu.mean()
        ic_gpu = 1.96 * cva_gpu.std(ddof=1) / np.sqrt(num_outer)

        diff = abs(m_cpu - m_gpu)
        ok = "OK" if diff < 2 * max(ic_cpu, ic_gpu) else "DIFF"
        rows.append([M_LS, f"{m_cpu:.2f} ±{ic_cpu:.2f}", f"{m_gpu:.2f} ±{ic_gpu:.2f}",
                     f"{diff:.2f}", f"{t_cpu:.2f}", f"{t_gpu:.2f}",
                     f"{t_cpu/t_gpu:.1f}x", ok])
    df = pd.DataFrame(rows, columns=["M_LS", "CPU value", "GPU value",
                                     "|CPU-GPU|", "t CPU (s)", "t GPU (s)", "speedup", "status"])
    print(f"\n CVA swaptions ({label}), M_cva={num_outer}, n_swpt={num_swaptions}")
    print(df.to_string(index=False))


 CVA swaptions (indicator), M_cva=256, n_swpt=5
 M_LS      CPU value      GPU value |CPU-GPU| t CPU (s) t GPU (s) speedup status
   16 292.52 ±112.88 300.64 ±117.10      8.12      0.84      0.03   32.2x     OK
   32 288.38 ±112.95 295.89 ±115.51      7.51      0.93      0.05   19.2x     OK
   64 279.02 ±108.87 282.62 ±109.93      3.60      1.10      0.09   11.8x     OK

 CVA swaptions (intensity), M_cva=256, n_swpt=5
 M_LS     CPU value     GPU value |CPU-GPU| t CPU (s) t GPU (s) speedup status
   16 316.40 ±12.16 316.44 ±12.17      0.05    719.35      1.07  673.9x     OK
   32 305.54 ±12.27 305.37 ±12.23      0.16    800.67      2.13  376.2x     OK
   64 299.01 ±12.31 298.93 ±12.34      0.08    960.82      4.27  225.3x     OK
